# NarratoAI Segment Event Benchmark

This notebook turns videos attached under `/kaggle/input` into Kaggle Benchmark cases.

Flow:

1. Scan dataset videos, normally `part4.mp4`.
2. Split the video into overlapping short segments.
3. Build one storyboard image per segment.
4. Define `narrato_segment_understanding` as a Kaggle Benchmark task.
5. Use `%choose narrato_segment_understanding` in Kaggle Benchmark UI to run models.
6. If `narrato_segment_understanding_report.jsonl` exists, export `candidate_clips.json` and NarratoAI script JSON.


In [ ]:
# =========================
# Cell 0. Environment
# =========================

import sys
import subprocess
import importlib.util


def pip_install(package: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


# protobuf must be installed before importing kaggle_benchmarks.
pip_install("protobuf==5.29.6")

for package, import_name in [
    ("opencv-python-headless", "cv2"),
    ("pillow", "PIL"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("tqdm", "tqdm"),
    ("kaggle-benchmarks", "kaggle_benchmarks"),
    ("jupyter_bokeh", "jupyter_bokeh"),
]:
    if importlib.util.find_spec(import_name) is None:
        pip_install(package)

print("Environment ready")


In [ ]:
# =========================
# Cell 1. Settings
# =========================

# Empty string means all videos. For the current benchmark dataset, use part4.mp4.
TARGET_VIDEO_NAME = "part4.mp4"

SEGMENT_SECONDS = 14.0
SEGMENT_STRIDE_SECONDS = 10.0
MAX_SEGMENTS_PER_VIDEO = None  # None = scan the full selected video
SAMPLE_FRAMES_PER_SEGMENT = 8
MIN_EVENT_SCORE = 7.0

# If Kaggle injects kbench.llm, this can run immediately. In ordinary kernels it will stay ready for %choose.
AUTO_EVALUATE_IF_MODEL_PROXY = True

TARGET_DURATION_SECONDS = [60, 90]


In [ ]:
# =========================
# Cell 2. Imports and paths
# =========================

import cv2
import json
import math
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from tqdm import tqdm

# pyright: reportMissingImports=false
import kaggle_benchmarks as kbench  # type: ignore[reportMissingImports]
from kaggle_benchmarks.content_types import images  # type: ignore[reportMissingImports]

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
STORYBOARD_DIR = WORK_ROOT / "storyboards"
RESULT_DIR = WORK_ROOT / "results"
SEGMENT_REPORT_JSONL = RESULT_DIR / "narrato_segment_understanding_report.jsonl"

STORYBOARD_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_LLM = getattr(kbench, "llm", None)
print("Kaggle Benchmark LLM proxy:", "available" if BENCHMARK_LLM is not None else "not available")


In [ ]:
# =========================
# Cell 3. Video discovery
# =========================

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".webm"}


def find_video_files(input_root: Path = INPUT_ROOT, target_name: str = TARGET_VIDEO_NAME) -> List[str]:
    videos = sorted(str(p) for p in input_root.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
    target_name = (target_name or "").strip()
    if target_name:
        filtered = [p for p in videos if Path(p).name == target_name or Path(p).match(target_name)]
        if filtered:
            videos = filtered
    return videos


video_files = find_video_files()
if not video_files:
    raise FileNotFoundError(f"No video files found under {INPUT_ROOT}; TARGET_VIDEO_NAME={TARGET_VIDEO_NAME!r}")

print(f"Video count: {len(video_files)}")
for item in video_files:
    print(item)


In [ ]:
# =========================
# Cell 4. Video helpers
# =========================


def probe_video(video_path: str) -> dict:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    if total_frames <= 0:
        raise RuntimeError(f"Invalid video metadata: {video_path}")
    return {
        "video_path": video_path,
        "fps": fps,
        "total_frames": total_frames,
        "duration": total_frames / fps if fps else 0.0,
        "width": width,
        "height": height,
    }


def read_frame_at_time(cap, fps: float, sec: float):
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(float(sec) * fps)))
    ok, frame = cap.read()
    return frame if ok else None


def resize_keep_ratio(frame, target_width: int = 360):
    h, w = frame.shape[:2]
    scale = target_width / max(1, w)
    return cv2.resize(frame, (target_width, max(1, int(h * scale))))


def ts(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    ms = int(round((seconds - math.floor(seconds)) * 1000))
    total = int(math.floor(seconds))
    if ms >= 1000:
        total += 1
        ms -= 1000
    h = total // 3600
    m = (total % 3600) // 60
    s = total % 60
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


In [ ]:
# =========================
# Cell 5. Build benchmark cases
# =========================


def build_video_segments(video_files: list[str]) -> pd.DataFrame:
    rows = []
    for video_path in video_files:
        info = probe_video(video_path)
        duration = float(info["duration"])
        segment_index = 0
        start_sec = 0.0
        while start_sec < duration:
            end_sec = min(start_sec + SEGMENT_SECONDS, duration)
            if end_sec - start_sec >= 3:
                rows.append({
                    "video_path": video_path,
                    "source_video": Path(video_path).name,
                    "segment_index": segment_index,
                    "start_sec": round(start_sec, 2),
                    "end_sec": round(end_sec, 2),
                    "duration": round(end_sec - start_sec, 2),
                })
                segment_index += 1
            if MAX_SEGMENTS_PER_VIDEO is not None and segment_index >= MAX_SEGMENTS_PER_VIDEO:
                break
            start_sec += SEGMENT_STRIDE_SECONDS
    return pd.DataFrame(rows)


segment_df = build_video_segments(video_files)
if segment_df.empty:
    raise RuntimeError("No benchmark cases generated")

case_path = RESULT_DIR / "benchmark_cases.json"
case_path.write_text(segment_df.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")

print("Benchmark case count:", len(segment_df))
display(segment_df.head(30))
print("Benchmark cases:", case_path)


In [ ]:
# =========================
# Cell 6. Storyboard builder
# =========================


def make_segment_storyboard(
    video_path: str,
    start_sec: float,
    end_sec: float,
    segment_index: int,
    output_dir: Path = STORYBOARD_DIR,
    num_frames: int = SAMPLE_FRAMES_PER_SEGMENT,
    cell_width: int = 360,
    cols: int = 4,
) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    video_duration = total_frames / fps if fps else float(end_sec)
    start_sec = max(0.0, float(start_sec))
    end_sec = min(float(end_sec), video_duration)
    if end_sec <= start_sec:
        cap.release()
        raise RuntimeError(f"Invalid segment: start={start_sec}, end={end_sec}")

    seg_duration = end_sec - start_sec
    timestamps = [start_sec + seg_duration * (i + 1) / (num_frames + 1) for i in range(num_frames)]

    frames = []
    valid_timestamps = []
    for timestamp in timestamps:
        frame = read_frame_at_time(cap, fps, timestamp)
        if frame is None:
            continue
        frames.append(resize_keep_ratio(frame, target_width=cell_width))
        valid_timestamps.append(round(timestamp, 2))
    cap.release()

    if not frames:
        raise RuntimeError(f"No valid frames: {video_path}, start={start_sec}, end={end_sec}")

    max_h = max(frame.shape[0] for frame in frames)
    normalized = []
    for frame in frames:
        h, _ = frame.shape[:2]
        if h < max_h:
            frame = cv2.copyMakeBorder(frame, 0, max_h - h, 0, 0, cv2.BORDER_CONSTANT, value=(255, 255, 255))
        normalized.append(frame)

    rows = math.ceil(len(normalized) / cols)
    blank = np.ones_like(normalized[0]) * 255
    grid_rows = []
    for row in range(rows):
        row_imgs = []
        for col in range(cols):
            index = row * cols + col
            row_imgs.append(normalized[index] if index < len(normalized) else blank)
        grid_rows.append(np.hstack(row_imgs))

    storyboard = np.vstack(grid_rows)
    digest = hashlib.md5(f"{video_path}-{segment_index}-{start_sec}-{end_sec}".encode()).hexdigest()[:8]
    out_path = output_dir / f"{Path(video_path).stem}_seg{int(segment_index):04d}_{digest}.jpg"
    cv2.imwrite(str(out_path), storyboard)
    return {
        "storyboard_path": str(out_path),
        "timestamps": valid_timestamps,
        "start_sec": start_sec,
        "end_sec": end_sec,
    }


In [ ]:
# =========================
# Cell 7. Benchmark schema and task
# =========================

EVENT_TYPES = {
    "death_fail",
    "strong_reaction",
    "coop_command",
    "puzzle_progress",
    "live_interaction",
    "transition",
    "low_value",
}


@dataclass
class NarratoSegmentUnderstanding:
    summary: str
    event_type: str
    score: float
    confidence: float
    visual_evidence: str
    highlight_reason: str
    main_objects: list[str]
    main_actions: list[str]
    scene: str
    screen_text: str
    ost: int
    ost_reason: str
    narration: str
    recommended_start_sec: float
    recommended_end_sec: float


def clamp(value, low, high):
    return max(low, min(high, value))


def append_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def safe_int(value, default=2):
    try:
        return int(value)
    except Exception:
        return default


@kbench.task(
    name="NarratoAI Segment Event Understanding",
    description="Identify edit-worthy game/live-stream events from sampled video segment storyboards.",
)
def narrato_segment_understanding(
    llm,
    video_path: str,
    segment_index: int,
    start_sec: float,
    end_sec: float,
) -> dict:
    storyboard_info = make_segment_storyboard(
        video_path=video_path,
        start_sec=start_sec,
        end_sec=end_sec,
        segment_index=int(segment_index),
    )
    img = images.from_path(storyboard_info["storyboard_path"])

    prompt = f"""
你正在为 NarratoAI 做游戏/直播短视频剪辑识别。

当前视频片段时间范围：
start_sec = {start_sec}
end_sec = {end_sec}

下面这张图片是该片段内部按时间顺序抽取的关键帧拼图。
你的任务不是写普通摘要，而是判断这段是否值得剪进短视频。

请输出结构化结果：
1. summary: 用中文概括片段发生了什么。
2. event_type: 只能是 death_fail / strong_reaction / coop_command / puzzle_progress / live_interaction / transition / low_value。
3. score: 0-10 的剪辑价值分。低价值赶路/等待/菜单操作给 0-3；有明确视觉事件给 7+。
4. confidence: 0-1 的置信度。
5. visual_evidence: 必须写清楚可见画面证据。没有画面证据时写空字符串，并把 event_type 设为 low_value。
6. highlight_reason: 为什么它值得或不值得剪。
7. main_objects: 主要人物、物体、地点元素。
8. main_actions: 主要动作变化。
9. scene: 场景类型，例如游戏画面、菜单、过场、室内、影视片段等。
10. screen_text: 只提取原始视频画面真实文字，不要提取 storyboard 标签。
11. ost: NarratoAI 音频策略：0=纯 AI 解说，1=保留原声不加旁白，2=保留低音量原声同时叠加 AI 解说。
12. ost_reason: 一句话解释 OST。
13. narration: 如果适合加旁白，写一句 18-38 个中文字符的自然解说；如果必须保留原声，可写空字符串。
14. recommended_start_sec / recommended_end_sec: 推荐剪辑入点和出点，必须在当前片段范围内；好片段保留前后 1-3 秒。

重要规则：
- 不要只因为字幕或文字看起来有趣就判高分，必须有视觉证据。
- 菜单、加载、普通行走、无变化画面通常是 low_value。
- 死亡、失败、掉落、强反应、关键机关推进、明显合作动作才适合高分。
- 不要输出 Markdown，不要输出多余解释。
"""

    result = llm.prompt(prompt, image=img, schema=NarratoSegmentUnderstanding)

    event_type = str(result.event_type or "").strip()
    if event_type not in EVENT_TYPES:
        event_type = "low_value"

    score = clamp(float(result.score), 0.0, 10.0)
    confidence = clamp(float(result.confidence), 0.0, 1.0)
    rec_start = clamp(float(result.recommended_start_sec), float(start_sec), float(end_sec))
    rec_end = clamp(float(result.recommended_end_sec), float(start_sec), float(end_sec))
    if rec_end <= rec_start:
        rec_start, rec_end = float(start_sec), float(end_sec)

    visual_evidence = str(result.visual_evidence or "").strip()
    if not visual_evidence:
        event_type = "low_value"
        score = min(score, 3.0)

    ost = safe_int(result.ost, 2)
    if ost not in (0, 1, 2):
        ost = 2

    kbench.assertions.assert_true(event_type in EVENT_TYPES, expectation="event_type must be valid")
    kbench.assertions.assert_true(0.0 <= score <= 10.0, expectation="score must be in 0-10")
    kbench.assertions.assert_true(0.0 <= confidence <= 1.0, expectation="confidence must be in 0-1")

    record = {
        "video_path": video_path,
        "source_video": Path(video_path).name,
        "segment_index": int(segment_index),
        "start": float(start_sec),
        "end": float(end_sec),
        "duration": round(float(end_sec) - float(start_sec), 2),
        "recommended_start": round(rec_start, 2),
        "recommended_end": round(rec_end, 2),
        "recommended_duration": round(rec_end - rec_start, 2),
        "event_type": event_type,
        "score": score,
        "confidence": confidence,
        "visual_evidence": visual_evidence,
        "highlight_reason": result.highlight_reason,
        "ost": ost,
        "text": result.narration,
        "summary": result.summary,
        "scene": result.scene,
        "main_objects": result.main_objects,
        "main_actions": result.main_actions,
        "screen_text": result.screen_text,
        "ost_reason": result.ost_reason,
        "storyboard": storyboard_info["storyboard_path"],
        "sample_timestamps": storyboard_info["timestamps"],
    }
    append_jsonl(SEGMENT_REPORT_JSONL, record)
    return record


In [ ]:
# =========================
# Cell 8. Optional auto-evaluate when model proxy exists
# =========================

if AUTO_EVALUATE_IF_MODEL_PROXY and BENCHMARK_LLM is not None:
    if SEGMENT_REPORT_JSONL.exists():
        SEGMENT_REPORT_JSONL.unlink()
    eval_df = segment_df[["video_path", "segment_index", "start_sec", "end_sec"]].copy()
    print("Auto-evaluating benchmark cases:", len(eval_df))
    runs = narrato_segment_understanding.evaluate(
        llm=[BENCHMARK_LLM],
        evaluation_data=eval_df,
        max_attempts=2,
        retry_delay=5,
        remove_run_files=False,
    )
    display(runs.as_dataframe().head(30))
else:
    print("No Benchmark model proxy in this kernel. Use the final %choose cell in Kaggle Benchmark UI.")


In [ ]:
# =========================
# Cell 9. Export candidate_clips.json if results exist
# =========================


def load_segment_records(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


def export_candidate_clips(records: list[dict]):
    dedup = {}
    for record in records:
        key = (
            record.get("source_video") or Path(str(record.get("video_path", ""))).name,
            round(float(record.get("recommended_start", record.get("start", 0))), 2),
            round(float(record.get("recommended_end", record.get("end", 0))), 2),
        )
        old = dedup.get(key)
        if old is None or float(record.get("score", 0)) > float(old.get("score", 0)):
            dedup[key] = record

    records = sorted(
        dedup.values(),
        key=lambda item: (item.get("source_video", ""), float(item.get("recommended_start", item.get("start", 0)))),
    )
    selected = [
        record for record in records
        if float(record.get("score", 0)) >= MIN_EVENT_SCORE
        and record.get("event_type") != "low_value"
        and str(record.get("visual_evidence", "")).strip()
    ]

    candidate_clips = []
    script = []
    for index, record in enumerate(selected, start=1):
        start = float(record.get("recommended_start", record.get("start", 0)))
        end = float(record.get("recommended_end", record.get("end", start + 1)))
        timestamp = f"{ts(start)}-{ts(end)}"
        clip_id = f"clip_{index:04d}"
        narration = str(record.get("text", "") or "").strip()
        ost = safe_int(record.get("ost", 1 if not narration else 2), 2)
        if ost in (0, 2) and not narration:
            ost = 1

        candidate_clips.append({
            "clip_id": clip_id,
            "source_event_ids": [f"seg_{int(record.get('segment_index', index)):04d}_{record.get('event_type', 'event')}"],
            "timestamp": timestamp,
            "role": record.get("event_type", "unknown"),
            "score": float(record.get("score", 0)),
            "confidence": float(record.get("confidence", 0)),
            "picture": record.get("visual_evidence") or record.get("summary", ""),
            "visual_evidence": record.get("visual_evidence", ""),
            "highlight_reason": record.get("highlight_reason", ""),
            "narration_hint": narration,
            "OST": ost,
            "source_video": record.get("source_video") or Path(str(record.get("video_path", ""))).name,
            "storyboard": record.get("storyboard", ""),
        })
        script.append({
            "_id": clip_id,
            "timestamp": timestamp,
            "picture": record.get("visual_evidence") or record.get("summary", ""),
            "narration": narration,
            "OST": ost,
            "score": float(record.get("score", 0)),
            "event_type": record.get("event_type", "unknown"),
            "visual_evidence": record.get("visual_evidence", ""),
            "source_video": record.get("source_video") or Path(str(record.get("video_path", ""))).name,
        })

    candidate_path = RESULT_DIR / "candidate_clips.json"
    script_path = RESULT_DIR / "narrato_segment_script_converted.json"
    legacy_script_path = RESULT_DIR / "narrato_segment_script.json"

    candidate_path.write_text(
        json.dumps({"target_duration_seconds": TARGET_DURATION_SECONDS, "clips": candidate_clips}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    script_path.write_text(json.dumps(script, ensure_ascii=False, indent=2), encoding="utf-8")
    legacy_script_path.write_text(json.dumps(script, ensure_ascii=False, indent=2), encoding="utf-8")
    return candidate_path, script_path, len(records), len(candidate_clips)


records = load_segment_records(SEGMENT_REPORT_JSONL)
if records:
    candidate_path, script_path, total_count, clip_count = export_candidate_clips(records)
    print("Segment records:", total_count)
    print("Candidate clips:", clip_count)
    print("candidate_clips:", candidate_path)
    print("NarratoAI script:", script_path)
else:
    print("No segment report yet:", SEGMENT_REPORT_JSONL)
    print("Run the Benchmark task first, then rerun this cell to export clips.")


In [ ]:
%choose narrato_segment_understanding
